# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://konga.com")
links

['/content/back-to-school',
 '/content/back-to-school',
 '/',
 'https://shq.konga.com/register',
 '/stores',
 '/',
 '/account/profile',
 '/cart',
 '/',
 'tel:07080635700',
 'https://wa.me/+2347016542481',
 'mailto:help@konga.com?Subject=Hello%20',
 'tel:07080635700',
 '/help/contact-us',
 'https://group.konga.com/about.html',
 'https://group.konga.com/career.html',
 'https://affiliate.konga.com/',
 'https://blog.konga.com',
 '/content/terms-and-conditions',
 'https://www.kongapay.com',
 '/account/wallet',
 'https://www.myverveworld.com',
 'https://www.mastercard.com',
 'https://www.visa.com',
 '/content/buyer-safety',
 '/content/faqs',
 '/content/delivery',
 '/content/return-policy',
 '/content/bulk-purchase',
 '/sitemap.xml',
 'https://track.konga.com',
 '/content/privacy-policy',
 '/content/authentic-items-policy',
 '/get-mobile-app',
 'https://play.google.com/store/apps/details?id=com.konga.androida',
 'https://itunes.apple.com/us/app/konga/id880918394?ls=1&mt=8',
 'https://www.face

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'external company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [13]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'resume page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'code repository', 'url': 'https://github.com/huggingface'},
  {'type': 'social media', 'url': 'https://twitter.com/huggingface'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/company/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
MiniMaxAI/MiniMax-H3
Updated
about 11 hours ago
•
10.8k
•
2.52k
deepseek-ai/DeepSeek-V4-Flash-0731
Updated
5 days ago
•
433k
•
2.5k
moonshotai/Kimi-K3
Updated
9 days ago
•
1.13M
•
10.1k
Comfy-Org/MiniMax-H3
Updated
about 1 hour ago
•
2
•
761
DavidAU/Qwen3.6-27B-Fable-Fusion-711-Uncens

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nMiniMaxAI/MiniMax-H3\nUpdated\nabout 11 hours ago\n•\n10.8k\n•\n2.52k\ndeepseek-ai/DeepSeek-V4-Flash-0731\nUpdated\n

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community and platform focused on building the future of machine learning. It serves as the central hub where researchers, developers, and organizations collaborate to create, share, and innovate with machine learning models, datasets, and applications. 

The Hugging Face ecosystem hosts over **2 million models**, **500,000+ datasets**, and **1 million+ AI applications**, empowering users worldwide to contribute to and leverage state-of-the-art AI technologies.

---

## What We Offer

- **Model Hub:** Access to a vast library of pre-trained models spanning NLP, computer vision, audio, and more. Models are community-driven and continuously updated.
  
- **Datasets:** Extensive collections of datasets curated for diverse ML tasks, with the ability to host and collaborate on new datasets.
  
- **Spaces:** A feature to create and share interactive machine learning apps running seamlessly on the platform.
  
- **HuggingChat:** Hugging Face’s own AI-powered chat interface, showcasing conversational AI advancements.
  
- **Enterprise Solutions:** Tailored enterprise support, including model inference endpoints, providers, storage buckets, and service plans designed for teams and businesses.
  
- **Learning & Community:** Rich resources including documentation, blogs, daily research papers, forums, Discord channels, and GitHub repositories foster continuous learning and open collaboration.

---

## Our Community and Customers

Hugging Face’s community consists of AI researchers, data scientists, software developers, startups, and large enterprises. The platform facilitates collaboration on open-source AI projects and provides tools that allow businesses to integrate AI solutions efficiently. 

Customers range from academic researchers utilizing datasets and models for experimentation to enterprises deploying scalable AI inference with professional support.

---

## Company Culture

Hugging Face thrives on collaboration, openness, and innovation. The company sees itself as a community builder, encouraging contributions and knowledge sharing that advance the entire machine learning ecosystem.

- **Open Source & Transparency:** Most resources and tools are open to all, fostering trust and collective growth.
- **Innovation-Driven:** Continuous updates and new feature launches keep the platform at the forefront of AI.
- **Inclusive Community:** Engagement through forums, Discord, and daily papers ensures everyone has a voice.
- **Developer-Centric:** Easy-to-use APIs and integration tools empower developers to build and experiment seamlessly.

---

## Careers at Hugging Face

Hugging Face is actively growing and invites talented individuals passionate about AI and open collaboration to join the team. Opportunities span AI research, engineering, product development, and community management.

- Work on cutting-edge machine learning technologies.
- Collaborate with a vibrant global AI community.
- Contribute to open-source projects impacting millions globally.
- Enjoy a diverse, inclusive, and remote-friendly work environment.

**Join us to help build the future of AI!**

---

## Contact & Engage

- **Website:** [huggingface.co](https://huggingface.co)
- **Community Forum:** Connect with experts and enthusiasts
- **Discord & GitHub:** Join conversations and contribute to projects
- **Enterprise Inquiries:** Tailored support for business needs

---

## Brand Highlights

- Official colors: Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- Distinctive smiling face logo symbolizing a friendly and collaborative AI community

---

Hugging Face is where machine learning innovation meets community power—come explore, create, and collaborate!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a collaborative platform where researchers, developers, and enterprises come together to create, discover, and share machine learning models, datasets, and applications. Hugging Face is recognized as the home of machine learning innovation and community collaboration, fostering an open ecosystem of tools and resources that empower AI development worldwide.

---

## Our Platform & Offerings

- **Models Hub:** Browse and contribute over **2 million machine learning models** covering numerous AI tasks, including natural language processing, computer vision, and more.
- **Datasets Repository:** Access and collaborate on over **500,000 datasets**, all hosted and maintained by the community.
- **Spaces:** Run and share thousands of machine learning-powered applications and demos through Hugging Face's hosted app platform.
- **Buckets & Storage:** Secure cloud storage options optimized for ML workloads.
- **HuggingChat:** An AI-powered conversational interface designed for seamless interactions.
- **Enterprise Solutions:** Custom support and integration options for businesses with Hugging Face PRO, Inference Providers, and dedicated Enterprise Support.
- **Learning and Community:** Extensive documentation, blog posts, daily research paper highlights, and an active community across Discord, forums, and GitHub.

---

## Customers & Users

Hugging Face serves a diverse range of users including:

- **AI Researchers** who contribute cutting-edge models and datasets.
- **Machine Learning Engineers and Developers** building real-world AI applications.
- **Enterprises** integrating state-of-the-art AI with premium support and services.
- **Educators and Students** exploring machine learning through practical resources and tools.
- The platform supports millions of users globally collaborating on open-source projects and innovation.

---

## Company Culture

Hugging Face fosters an **open, collaborative, and inclusive culture** where creativity and knowledge-sharing drive progress. The company is deeply committed to transparency, supporting an active community that contributes to and benefits from collective AI advancements. Team members thrive in a dynamic, mission-driven environment passionate about expanding access to ML technologies and empowering innovators everywhere.

---

## Careers at Hugging Face

Join Hugging Face and help build the future of AI! The company is continually growing and looking for talent in areas such as:

- Machine learning research
- Software engineering and platform development
- Community management and developer advocacy
- Enterprise solutions and customer success

Employees enjoy the opportunity to work with state-of-the-art technology in an environment that values autonomy, creativity, and impact. Hugging Face welcomes collaborators who are passionate about AI and open source.

---

## Why Hugging Face?

- **Leading AI Collaboration Hub:** Join the largest and most active machine learning community.
- **Open & Open-Source:** Empower AI breakthroughs through shared models, data, and solutions.
- **Cutting-edge Tools:** Access versatile platforms for building, deploying, and managing AI models.
- **Enterprise-ready:** Robust solutions and support for business-grade AI deployment.
- **Community-driven Innovation:** Continuous growth fueled by research, open discussions, and collaboration.

---

### Connect with Us

- **Website:** https://huggingface.co
- **Community:** Discord, GitHub, Forum
- **Blog & Learning Resources:** AI research updates, tutorials, and papers.

---

**Hugging Face — The AI community building the future.**  
Your partner in advancing machine learning, together.

---

*Brand colors: #FFD21E (yellow), #FF9D00 (orange), #6B7280 (gray)*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>